# 8.6 Word embeddings (Word2Vec, GloVe, FastText)

Word embeddings make the distributional hunch geometric: words that live in similar neighborhoods learn vectors that point in similar directions.

This notebook simulates embeddings with inline co-occurrence counts, PPMI weighting, and SVD. It also shows why cosine normalization and subword evidence matter on rare or frequency-skewed words.

Save a copy to Drive to edit.

In [ ]:

import math
import random
import re
import unicodedata
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(8)
np.random.seed(8)


def exact_accuracy(predictions, expected):
    correct = 0
    for pred, gold in zip(predictions, expected):
        if pred == gold:
            correct += 1
    return correct / max(1, len(expected))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def tokenize_doc(doc):
    return re.findall(r"[a-z0-9]+", doc.casefold())


def show_matrix(ax, matrix, title, xlabels=None, ylabels=None):
    image = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if xlabels is not None:
        ax.set_xticks(range(len(xlabels)))
        ax.set_xticklabels(xlabels, rotation=45, ha="right")
    if ylabels is not None:
        ax.set_yticks(range(len(ylabels)))
        ax.set_yticklabels(ylabels)
    return image


## The concept, built once (D1): co-occurrence to PPMI/SVD embeddings

Skip-gram writes $P(c\mid w)=\frac{\exp(u_c^\top v_w)}{\sum_{j\in V}\exp(u_j^\top v_w)}$, while count-based methods chase similar geometry through co-occurrence. The lesson count matrix has row sums 10 and total 40; for `king` with `man`, $\log((5\cdot40)/(10\cdot10))=\log 2=0.693$.

In [ ]:

def ppmi_from_counts(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    row_sums = counts.sum(axis=1, keepdims=True)
    col_sums = counts.sum(axis=0, keepdims=True)
    ppmi = np.zeros_like(counts)
    for i in range(counts.shape[0]):
        for j in range(counts.shape[1]):
            if counts[i, j] == 0:
                continue
            ratio = counts[i, j] * total / (row_sums[i, 0] * col_sums[0, j])
            ppmi[i, j] = max(0.0, math.log(ratio))
    return ppmi


def cooc_to_svd_embeddings(counts, words, dim=2):
    ppmi = ppmi_from_counts(counts)
    u, s, vt = np.linalg.svd(ppmi, full_matrices=False)
    vectors = u[:, :dim] * np.sqrt(s[:dim])
    return {word: vectors[i] for i, word in enumerate(words)}, ppmi

words = ["king", "queen", "man", "woman"]
counts = np.array([
    [0, 0, 5, 5],
    [0, 0, 5, 5],
    [5, 5, 0, 0],
    [5, 5, 0, 0],
], dtype=float)
embeddings, ppmi = cooc_to_svd_embeddings(counts, words, dim=2)
king_man_ppmi = ppmi[words.index("king"), words.index("man")]

assert np.allclose(counts.sum(axis=1), [10, 10, 10, 10])
assert counts.sum() == 40
assert counts[words.index("king"), words.index("man")] == 5
assert abs(king_man_ppmi - math.log(2)) < 1e-12


The exact lesson cosine and analogy numbers are separate toy geometry checks: vector direction can be audited with cosine, and relations can be represented as offsets.

In [ ]:

target_cos = 0.9986178293325095
angle = math.acos(target_cos)
king_vec = np.array([1.0, 0.0])
queen_vec = np.array([math.cos(angle), math.sin(angle)])
man_vec = np.array([math.cos(angle), -math.sin(angle)])
car_vec = -king_vec

toy_cosines = {
    "king_queen": cosine(king_vec, queen_vec),
    "king_man": cosine(king_vec, man_vec),
    "king_car": cosine(king_vec, car_vec),
}

analogy_king = np.array([2, 1])
analogy_man = np.array([1, 0])
analogy_woman = np.array([1, 2])
analogy_queen = analogy_king - analogy_man + analogy_woman

assert abs(toy_cosines["king_queen"] - 0.9986178293325095) < 1e-12
assert abs(toy_cosines["king_man"] - 0.9986178293325095) < 1e-12
assert abs(toy_cosines["king_car"] - (-1.0)) < 1e-12
assert analogy_queen.tolist() == [2, 3]


## Dataset ladder: inline co-occurrence corpora from toy royal words to noisy rare-word data

In [ ]:

embedding_ladder = [
    {
        "name": "D1 lesson royal-word toy",
        "sentences": ["king man royal", "king woman royal", "queen woman royal", "queen man royal"],
        "queries": [("king", "queen")],
    },
    {
        "name": "D2 five clean analogy pairs",
        "sentences": ["paris france capital", "rome italy capital", "berlin germany capital", "madrid spain capital", "queen king royal"],
        "queries": [("paris", "rome"), ("france", "italy")],
    },
    {
        "name": "D3 polysemy frequency morphology",
        "sentences": ["play game fun", "played game yesterday", "playing game today", "bank river water", "bank loan money", "money loan bank"],
        "queries": [("play", "playing"), ("loan", "money")],
    },
    {
        "name": "D4 tiny inline category corpus",
        "sentences": ["campaign click bid ads", "budget bid campaign ads", "creator video content", "video sponsor creator", "invoice receipt billing", "payment invoice billing"],
        "queries": [("campaign", "ads"), ("invoice", "billing"), ("creator", "video")],
    },
    {
        "name": "D5 longer biased rare-word corpus",
        "sentences": [
            "engineer system code code code",
            "nurse care patient care care",
            "creator video sponsor revenue",
            "creators video sponsors revenue",
            "play playing played replay",
            "rarewordx campaign niche",
            "campaign click budget budget budget",
        ],
        "queries": [("creator", "creators"), ("play", "played"), ("campaign", "budget")],
    },
]

for rung in embedding_ladder:
    print(rung["name"], "sentences", len(rung["sentences"]), "queries", len(rung["queries"]))
    print("sample", rung["sentences"][0])


In [ ]:

def cooc_from_sentences(sentences, window=2):
    tokenized = [tokenize_doc(sentence) for sentence in sentences]
    vocabulary = sorted({token for sent in tokenized for token in sent})
    index = {word: i for i, word in enumerate(vocabulary)}
    counts = np.zeros((len(vocabulary), len(vocabulary)), dtype=float)
    for sent in tokenized:
        for i, word in enumerate(sent):
            left = max(0, i - window)
            right = min(len(sent), i + window + 1)
            for j in range(left, right):
                if i == j:
                    continue
                counts[index[word], index[sent[j]]] += 1
    return counts, vocabulary


def nearest_neighbor_accuracy(sentences, queries):
    counts, vocabulary = cooc_from_sentences(sentences)
    vectors, ppmi = cooc_to_svd_embeddings(counts, vocabulary, dim=min(3, len(vocabulary)))
    predictions = []
    expected = []
    for query, gold in queries:
        if query not in vectors or gold not in vectors:
            predictions.append(None)
            expected.append(gold)
            continue
        scores = []
        for word in vocabulary:
            if word == query:
                continue
            scores.append((cosine(vectors[query], vectors[word]), word))
        scores.sort(reverse=True)
        predictions.append(scores[0][1])
        expected.append(gold)
    return exact_accuracy(predictions, expected), vectors, ppmi, vocabulary

embedding_results = []
for index, rung in enumerate(embedding_ladder, start=1):
    acc, vectors, ppmi, vocabulary = nearest_neighbor_accuracy(rung["sentences"], rung["queries"])
    embedding_results.append({"rung": index, "name": rung["name"], "accuracy": acc, "vectors": vectors, "ppmi": ppmi, "vocab": vocabulary})

for row in embedding_results:
    print(row["rung"], row["name"], f"nn_accuracy={row['accuracy']:.3f}", "vocab", len(row["vocab"]))


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(embedding_results):
    vocab = row["vocab"][:8]
    matrix = np.zeros((len(vocab), len(vocab)))
    for i, a in enumerate(vocab):
        for j, b in enumerate(vocab):
            matrix[i, j] = cosine(row["vectors"][a], row["vectors"][b])
    show_matrix(axes[0, idx], matrix, f"D{idx + 1} cosine", vocab, vocab)

x = [row["rung"] for row in embedding_results]
y = [row["accuracy"] for row in embedding_results]
axes[1, 0].plot(x, y, marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title("NN accuracy vs vocabulary noise")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("accuracy")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on D5: raw dot products and over-sold analogies

In [ ]:

def char_trigrams(word):
    if len(word) < 3:
        return {word}
    return {word[i:i + 3] for i in range(len(word) - 2)}


def jaccard(a, b):
    left = char_trigrams(a)
    right = char_trigrams(b)
    return len(left & right) / len(left | right)

d5 = embedding_results[-1]
vectors = d5["vectors"]
query = "campaign"
raw_scores = sorted((float(np.dot(vectors[query], vec)), word) for word, vec in vectors.items() if word != query)
cos_scores = sorted((cosine(vectors[query], vec), word) for word, vec in vectors.items() if word != query)
play_playing = jaccard("play", "playing")
play_played = jaccard("play", "played")

print("raw dot top", raw_scores[-1])
print("cosine top", cos_scores[-1])
print("FastText-style play/playing overlap", play_playing)
print("FastText-style play/played overlap", play_played)
print("toy analogy queen point", analogy_queen.tolist())


## Evaluate it + Practice

- Metric: nearest-neighbor or analogy accuracy from cosine similarities
- No-skill baseline: one-hot vectors where every different word has zero cosine
- Cheap sanity check: D1 PPMI king-man must be log 2, and toy king-car cosine must be -1
- Ablation: rank with raw dot products or remove subword overlap and D5 rare forms degrade
- Failure signals: frequency-dominated neighbors, biased neighborhoods, or analogies treated as truth

### Practice

**Practice:** Add one rare morphological variant to D5 and compute its trigram overlap with the base word.

**Practice:** Compare SVD dimensions 2 and 3 on the D4 nearest-neighbor queries.